<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">05 Mobile AI</h1>
<hr>


## **2. Understanding Word Embedding**

<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)
  
</div>


### 2.1 Introduction
In this hands-on exercise, we explore how language models represent meaning through semantic vectors and how these vectors can be compared using what is referred to as **cosine similarity**. We examine classic examples such as the analogy “king - man + woman ≈ queen” to illustrate how embedding spaces encode conceptual relationships. Using a small model (*all-MiniLM-L6-v2*) we generate word embeddings, compute similarities and nearest neighbors. We will also apply simple vector arithmetic to reveal structural patterns in the semantic space.

#### 2.1.1 What Are Embeddings?

Embeddings are numerical representations of words, sentences, or even entire documents that capture their semantic meaning. Instead of treating language as discrete symbols, an embedding model maps each element (like a word) to a point in a continuous vector space. These typically consist of hundreds or thousands of dimensions. In this vector space, words with similar meanings are located close to each other. Relationships such as analogies (“Paris” is to “France” as “Rome” is to “Italy”) emerge as simple geometric patterns. The distance and direction between vectors encode information about context and meaning. Embeddings are foundational as they allow mathematical operations on language. They are one of the core concepts enabling models to reason about similarity, analogy, and context.

#### 2.1.2 Cosine Similarity

When working with word embeddings or language model representations, each word (or token) is represented as a high-dimensional vector. These vectors capture semantic relationships: words with similar meanings tend to have vectors that point in similar directions in this multidimensional space. To quantify how similar two embeddings are, experts use what is called **Cosine Similarity**. Unlike Euclidean distance, which measures how far apart two points are, cosine similarity measures how aligned their directions are, regardless of their magnitudes. This makes it particularly useful for comparing word vectors, where only the direction (semantic meaning) matters, not the length.

The Cosine Similarity between two embedding vectors A and B is defined as:

$$
\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{||A|| ~ ||B||}
$$

Where $A \cdot B = \sum_i A_i B_i$ is the dot product of the two vectors, and $||A|| = \sqrt{\sum_i A_i^2}$ and $||B|| = \sqrt{\sum_i B_i^2}$ are the Euclidean norms (the lengths) of the two vectors. The denominator term $||A|| ~ ||B||$ is an ordinary scalar product of the magnitudes. It is used to normalize the result.

Cosine Similarity measures the **angle** between vectors, not their length. A value of **1.0** indicates that two vectors point in the same direction (very similar, like "parent" and "father"). The value **0.0** means that vectors are orthogonal (unrelated, like "king" and "tomato"). A value of **-1.0** indicates that the vectors point in opposite directions and represent true opposites. The value **-1.0** is especially interesting. Embeddings capture meaning through contextual co-occurrence rather than strict semantic opposition. Words like "hot" and "cold" frequently appear in similar contexts (e.g., weather, temperature), so they are somewhat related, not perfectly opposed. A stronger example of oppositeness could be numerical counterparts such as "100" and "-100".

#### 2.1.3 The model [`all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)

The model `all‑MiniLM‑L6‑v2` converts sentences and paragraphs into 384-dimensional vector embeddings. These representations support tasks such as finding similar text (**semantic search**) and grouping related texts (**clustering**). The model [is based on a compact MiniLM architecture](https://codemaker2016.medium.com/building-a-symptoms-based-diagnosis-system-using-all-minilm-l6-v2-211411982fde) and fine-tuned specifically for high-quality sentence embeddings. It delivers strong semantic similarity performance while remaining fast on standard CPUs, all while requiring only a small memory footprint. The default maximum input length is 256 tokens. Models of this type are often referred to as **Small Language Models**.



<div class="alert alert-block alert-success">

Make sure that you have installed all necessary packages. If you are missing some, you can either use `pip install` or the provided cell in [00.1_Check_Course_Environment.ipynb](../00_Intro/00.1_Check_Course_Environment.ipynb) (run the cell in section 1.2: "Install the Course Libraries").
</div>

### 2.2 Sentence Transformer model
We start by initializing the model and printing the vector space dimension to ensure functionality.

In [ ]:
from sentence_transformers import SentenceTransformer

# Small, fast model for classroom demos
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # ~384-dim, fast
model = SentenceTransformer(MODEL_NAME)
dim = model.get_embedding_dimension()
dim

For the rest of the notebook, we will need some helper functions to work with the model:

In [ ]:
import numpy as np
from numpy.linalg import norm as l2norm
import pandas as pd

# --- Globals ---
EMB_MEAN = None
_EPS = 1e-9

# --- Helpers ---
def _l2_normalize_rows(E, eps=_EPS):
    # row-wise L2 normalization
    n = l2norm(E, axis=1, keepdims=True)
    return E / (n + eps)

def _safe_unit(v, eps=_EPS):
    # zero-vector guard
    n = l2norm(v)
    return v if n <= eps else v/n

def _index(word, vocab):
    try:
        return vocab.index(word)
    except ValueError:
        raise KeyError(f"Word not in vocabulary: {word}")
    

# --- Core ---
def embed(words, fit_mean=False):
    # return array of embeddings
    global EMB_MEAN
    # get *unnormalized* embeddings from the model
    E = np.asarray(model.encode(words, normalize_embeddings=False), dtype=np.float32)

    if fit_mean:
        EMB_MEAN = E.mean(axis=0, keepdims=True)
    if EMB_MEAN is not None:
        E = E - EMB_MEAN
    
    return _l2_normalize_rows(E)

def cosine(a, b):
    # cosine similarity of two L2-normalized 1-D vectors
    return float(np.dot(a, b))

def most_similar(query_vec, vocab_vecs, vocab_words, topk=5, exclude=set()):
    # return top-k nearest words by cosine
    sims = vocab_vecs @ query_vec
    rank = np.argsort(-sims)

    out = []
    for idx in rank:
        w = vocab_words[idx]
        if w in exclude:
            continue
        out.append((w, float(sims[idx])))
        if len(out) >= topk:
            break
    return out

def analogy(a, b, c, V, vocab, exclude=None, topk=5):
    # solve a : b :: c : ?
    A = V[_index(a, vocab)]
    B = V[_index(b, vocab)]
    C = V[_index(c, vocab)]

    q = _safe_unit(B - A + C)

    base_excl = {a, b, c}
    if exclude is not None:
        base_excl |= set(exclude)

    return most_similar(q, V, vocab, topk, base_excl)


# --- Utility wrappers ---

def vec(word):
    return V[vocab.index(word)]

def iter_cosine(pairs):
    rows = []
    for a, b in pairs:
        rows.append({
            "a": a, 
            "b": b,
            "cosine-similarity(a,b)": cosine(vec(a), vec(b))
        })
    return rows

### 2.3 Test similarities of an example mini vocabulary

We now embed a small, interpretable set of words and test their cosine similarities.

#### 2.3.1 Vocabulary and embeddings


In [ ]:
vocab = [
    "father", "mother", "man", "woman", "boy", "girl", "son", "daughter",
    "king", "queen", "prince", "princess", "duke", "duchess",
    "doctor", "nurse", "engineer", "teacher", "lawyer", "chef", "scientist", "artist",
    "paris", "london", "zurich", "switzerland", "bern", "france", "japan", "germany",
    "car", "bicycle", "apple", "orange", "cat", "dog", "river", "mountain", "good", 
    "bad", "cold", "hot", "minus", "plus", "true", "false", "banana",
    "-100", "100", "spaghetti", "bratwurst", "sushi", "quantum", "42", "teacher", "democracy",
    "screwdriver", "love", "refrigerator", "switzerland","running", "python", "cell", "prayer", "algorithm", "italy", "pizza", "pasta"
]

V = embed(vocab, fit_mean=True)
count, dims = V.shape
print(f"{count} items embedded over {dims} dimensions")

#### 2.3.2 Similarity table

We now compute a similarity table for a few examples, and observe the similarities among the paired words.


In [ ]:
pairs = [
    ("father", "mother"),
    ("king", "queen"),
    ("doctor", "nurse"),
    ("cat", "dog"),
    ("switzerland", "zurich"),
    ("switzerland", "london"),
    ("good", "bad"),
    ("cold", "hot"),
    ("-100", "100"),
    ("minus", "plus"),
    ("true", "100"),
    ("zurich", "banana"),
]

rows = iter_cosine(pairs)

df_sims = pd.DataFrame(rows)
df_sims

#### 2.3.3 Opposites in sentence-transformers
 
Getting cosine similarities below -0.6 (strong opposites) in sentence-transformer models is actually quite rare, even after recentering. Sentence-transformers are trained using contrastive loss functions that bring embeddings of semantically similar sentence pairs closer together. As a result, even antonyms (e.g. "good" - "bad") often end up with cosine values around +0.3-0.5 because they appear in similar linguistic contexts. Truly negative values (e.g. < -0.3) usually only appear between semantically distant or domain-disconnected terms. This could be mixing countries with fruits, or numbers with emotions.

Pairs likely to produce genuinely negative cosine similarities include:

| Category                     | Example pair                    | Explanation                                   |
| ---------------------------- | ------------------------------- | --------------------------------------------- |
| Completely unrelated domains | `quantum` - `banana`        | science vs. fruit, far apart in vector space |
| Abstract vs. concrete        | `democracy` - `screwdriver` | concept vs. tool                              |
| Positive emotion vs. object  | `love` - `refrigerator`     | emotional vs. physical                        |
| Country vs. verb             | `switzerland` - `running`   | entity vs. action                             |
| Human vs. number             | `teacher` - `42`            | semantic types do not match                    |
| Programming vs. biology      | `python` - `cell`           | totally different contexts                    |
| Religion vs. technology      | `prayer` - `algorithm`      | abstract vs. technical                        |


We can test this out:


In [ ]:
pairs = [
    ("quantum", "banana"),
    ("love", "refrigerator"),
    ("democracy", "screwdriver"),
    ("prayer", "algorithm"),
    ("switzerland", "running"),
    ("teacher", "42")
]

rows = iter_cosine(pairs)

df_sims = pd.DataFrame(rows)
df_sims

We still do not see strong opposites. Sentence-transformers are trained to pull similar sentences together, not to push unrelated ones apart. Training pairs come from paraphrases (“a dog is running” ≈ “an animal is sprinting”). There is no negative training signal saying that “quantum” should be *opposite* to “banana”. So embeddings for unrelated concepts just drift toward orthogonality (≈ 0), not true opposition ( < -0.5).


#### 2.3.4 Visualization in 3D with Principal Component Analysis (PCA)

Embeddings live in hundreds of dimensions (like 768 or 1024). We use **Principal Component Analysis (PCA)** to visualize their structure in 3D, while preserving the most meaningful variance. As a result, related words should appear clustered together. PCA is a mathematical technique used to reduce the dimensionality of data while keeping as much of its variation (information) as possible. Word embeddings, for example, might live in a hundred- or thousand-dimensional space. We cannot visualize that directly, so PCA helps us find a simpler, approximate version of it in 3D. PCA finds the directions (axes) where the points vary the most, called principal components. Then it projects the data onto the top few components (typically 2 or 3), giving us a compressed view that still shows the main patterns and clusters.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import plotly.graph_objs as go

# --- PCA ---
pca = PCA(n_components=3)
V_3d = pca.fit_transform(V)

# --- 3D scatter plot ---
evr = pca.explained_variance_ratio_
axis_titles = [f"PCA {i+1} ({100*evr[i]:.1f}%)" for i in range(3)]

def lims(a, lo=1, hi=99, pad=0.05):
    low, high = np.percentile(a, [lo, hi])
    span = high - low
    return (low - pad*span, high + pad*span)

x_limits = lims(V_3d[:,0])
y_limits = lims(V_3d[:,1])
z_limits = lims(V_3d[:,2])

n_clusters = 6  # adjust freely (3-6 recommended)
km = KMeans(n_clusters=n_clusters, n_init="auto", random_state=0)
labels = km.fit_predict(V_3d)

palette = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA",
           "#FFA15A", "#19D3F3", "#FF6692", "#B6E880",
           "#FF97FF", "#FECB52"]
colors = [palette[i % len(palette)] for i in labels]

BG = "#1e1e1e"
GRID = "#3a3a3a"
FONT = "#e5e5e5"

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=V_3d[:,0],
    y=V_3d[:,1],
    z=V_3d[:,2],
    mode="markers+text",
    text=vocab,
    textposition="top center",
    marker=dict(size=6, color=colors, opacity=0.94),
    hovertemplate="<b>%{text}</b><br>PCA1=%{x:.3f}<br>PCA2=%{y:.3f}<br>PCA3=%{z:.3f}<extra></extra>",
    name="points",
    showlegend=False
))

zero = dict(mode="lines", line=dict(width=2, color="#bbbbbb"), showlegend=False)
fig.add_trace(go.Scatter3d(x=[x_limits[0], x_limits[1]], y=[0,0], z=[0,0], **zero))
fig.add_trace(go.Scatter3d(x=[0,0], y=[y_limits[0], y_limits[1]], z=[0,0], **zero))
fig.add_trace(go.Scatter3d(x=[0,0], y=[0,0], z=[z_limits[0], z_limits[1]], **zero))

fig.update_layout(
   height=900,
    margin=dict(l=0, r=0, t=50, b=0),
    uirevision="keep",

    scene=dict(
        xaxis=dict(title=axis_titles[0], range=x_limits, backgroundcolor=BG, gridcolor=GRID, color=FONT),
        yaxis=dict(title=axis_titles[1], range=y_limits, backgroundcolor=BG, gridcolor=GRID, color=FONT),
        zaxis=dict(title=axis_titles[2], range=z_limits, backgroundcolor=BG, gridcolor=GRID, color=FONT),
        camera=dict(eye=dict(x=1.6, y=1.6, z=0.9)),
    ),
    paper_bgcolor=BG,
    font=dict(color=FONT),
)

fig.update_layout(
    updatemenus=[dict(
        type="buttons",
        x=0.01, y=1.10, xanchor="left",
        buttons=[
            dict(label="Front", method="relayout",
                 args=[{"scene.camera": {"eye": {"x": 0, "y": 0, "z": 2.2}}}]),
            dict(label="Iso", method="relayout",
                 args=[{"scene.camera": {"eye": {"x": 1.6, "y": 1.6, "z": 0.9}}}]),
            dict(label="Top", method="relayout",
                 args=[{"scene.camera": {"eye": {"x": 0, "y": 2.2, "z": 0}}}]),
        ]
    )]
)

fig.show()

### 2.4 Classic word analogy
We now test $a : b :: c :$? (read as “a is to b as c is to what?”), which expresses an analogy. An analogy is a relationship comparison between pairs of things.

Below we test the classic: **king - man + woman ≈ queen**. A king is to a man as a queen is to a woman.


In [ ]:
a, b, c = "man", "king", "woman" # a:b :: c:?  -> expect "queen"
result = analogy(a, b, c, V, vocab, exclude={a,b,c}, topk=10)
result

We also note that in our mini vocabulary, Japan is to Sushi as Germany is to ... (see below)

In [ ]:
a, b, c = "japan", "sushi", "germany" # a:b :: c:?  -> a stereotype would be "bratwurst"
result = analogy(a, b, c, V, vocab, exclude={a,b,c}, topk=10)
result

We can also verify that **("japan" minus "sushi")** is similar to **("germany" minus "bratwurst")** by measuring the cosine similarity between these two difference vectors. If the embedding space of our model captures the “country-food” relationship well, then these difference vectors should point in roughly the same direction.


In [ ]:
rel_1 = vec("japan") - vec("sushi")
rel_2  = vec("germany") - vec("bratwurst")

# Normalize the relation vectors before cosine
rel_1 /= (l2norm(rel_1) + 1e-9)
rel_2  /= (l2norm(rel_2)  + 1e-9)

cos_rel = cosine(rel_1, rel_2)

cos_rel

Larger models would have provided a cosine similarity around 0.7-0.8 for vectors such as `(japan - sushi)` and `(germany - bratwurst)`. This reflects their ability to encode consistent relation structures, in this case the `country-national food` semantic relation. Smaller models struggle to form such stable relation vectors, which is why they produce weaker or noisier similarities.

### 2.5 Summarizing remarks
Cosine similarity measures how closely two vectors align in direction, rather than in magnitude. It provides a simple yet powerful way to assess semantic relatedness between words or sentences: vectors pointing in similar directions indicate similar meanings, while those pointing in opposite directions suggest semantic contrast. This measure forms the heart of many embedding-based systems, powering applications such as search, clustering, recommendation, and natural language understanding.

Vector analogies give us a fascinating way to see how language models capture relationships in text. When we compute expressions such as “king - man + woman ≈ queen,” we are probing whether the model encodes consistent geometric patterns that mirror meaningful linguistic or conceptual relationships. These analogies often work surprisingly well, showing that certain grammatical and semantic regularities emerge naturally from the geometry of the learned space. At the same time, they are not perfect: some relationships fail or behave unpredictably, reminding us that these models are approximations built on data rather than true reasoning systems. Moreover, their behavior reflects the biases, associations, and cultural patterns present in the text dataset from which they were trained.

It is also important to distinguish between sentence embeddings and token embeddings. Sentence embeddings, such as those produced by models in the Sentence-Transformers library, are designed to capture the semantic similarity between entire phrases, sentences, or paragraphs. They bring semantically equivalent expressions close together in vector space, even when the wording differs. Token embeddings, on the other hand, represent individual words or subword units that form the model’s vocabulary. While these vectors encode basic linguistic and contextual information, they are not optimized for comparing longer texts or for expressing the meaning of full sentences.

It might be worth reflecting on the ethical dimension of embeddings: since they learn directly from human-produced text, they inevitably capture cultural stereotypes and societal biases. By recognizing and discussing these issues, we gain a deeper appreciation of both the capabilities and the limitations of the powerful language representations that shape modern artificial intelligence.

### 2.6 Playground

With the following cells you can try out some of your own combinations for solving the questions:

In [ ]:
# compare words a and b
a = "italy"
b = "asteroid"

# embed using same fitted mean
new_vecs = embed([a, b], fit_mean=False)

# compute cosine similarity
opp = cosine(new_vecs[0], new_vecs[1])
print(f"Opposition (cosine) between '{a}' and '{b}': {opp:.3f}")

In [ ]:
# select words to solve a : b :: c : ?
b = "italy"
a = "spaghetti"
c = "sushi"

# collect missing tokens
needed = [w for w in (a, b, c) if w not in vocab]

# extend vocab only with missing words
if needed:
    vocab = vocab + needed
    V = embed(vocab, fit_mean=False)
else:
    # optionally reuse existing embeddings
    # or re-embed everything depending on your pipeline
    V = embed(vocab, fit_mean=False)

result, cos = analogy(a, b, c, V, vocab, exclude={a, b, c}, topk=3)[0]

print(f"{a} is to {b} as {c} is to {result}\ncosine: {cos}")


### 2.7 Further Reading and Literature

[01] Grant Sanderson. [**Visualizing transformers and attention**](https://www.youtube.com/watch?v=KJtZARuO3JY&t=1256s). Talk for TNG Big Tech Day '24. YouTube video lecture, 2023.

[02] Tom Krantz & Alexandra Jonker. [**What Is Cosine Similarity?**](https://www.ibm.com/think/topics/cosine-similarity) IBM Think article, 2025.

----
### 2.8 Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

**Try to find examples of strong oppositeness, with a cosine similarity <0.** Answer:

(add your answer here)

**Try to find examples of strong similarity, with a cosine similarity >0.6.** Answer:

(add your answer here)

**Try `prince - man + woman ≈ ?` What do you get (expect something near princess)?** Answer:

(add your answer here)

**Try a profession: `doctor - man + woman ≈ ?` What do you get? Discuss biases and ethical issues.** Answer:

(add your answer here)

**Does the model associate Italy with “pizza”, “pasta”, or something else? What does this reveal about cultural associations in embeddings and the model's training data?** Answer:

(add your answer here)

**Find the result of `hot is to cold as minus is to ?`. Does the result make sense in the context of opposites? Why might the model struggle to represent true negation?** Answer:

(add your answer here)

**Are positive and negative numbers recognized as opposites? Why or why not?** Answer:

(add your answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)